# Modified CV Output Parser

This keeps the original Mistral + LangChain StructuredOutputParser approach, but separates professional experience from projects and adds profile/contact/language/certification fields.

In [ ]:
!pip install -U langchain langchain-community langchain-core langchain-classic transformers==4.52.4 pypdf

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from langchain_classic.output_parsers import StructuredOutputParser, ResponseSchema
from pypdf import PdfReader
import torch
import json
import re

model_name = "mistralai/Mistral-Nemo-Instruct-2407"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

def generate_text(prompt, max_new_tokens=1800):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=12000)
    if torch.cuda.is_available():
        inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=0.0,
            pad_token_id=tokenizer.eos_token_id
        )
    generated = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)


In [ ]:
pdf_path = "/kaggle/input/datasets/ahmedsuma/cv-test-2/CV final 2.pdf"  # change to your CV path
reader = PdfReader(pdf_path)
cv_text = "\n\n".join((page.extract_text() or "") for page in reader.pages).strip()
print(cv_text[:5000])


In [ ]:
response_schemas = [
    ResponseSchema(name="full_name", description="Candidate full name."),
    ResponseSchema(name="email", description="Candidate email, or empty string."),
    ResponseSchema(name="phone", description="Candidate phone, or empty string."),
    ResponseSchema(name="linkedin", description="LinkedIn URL/profile, or empty string."),
    ResponseSchema(name="github", description="GitHub URL/profile, or empty string."),
    ResponseSchema(name="profile", description="Professional summary/about/profile, or empty string."),
    ResponseSchema(name="education", description="List of education records: degree, institution, location, start_date, end_date, details."),
    ResponseSchema(name="experience", description="List of professional work/internship records: job_title, company, location, start_date, end_date, description. Do not include projects."),
    ResponseSchema(name="projects", description="List of academic/personal projects: project_name, dates, technologies, description."),
    ResponseSchema(name="skills", description="List of technical/professional skills explicitly mentioned."),
    ResponseSchema(name="languages", description="List of spoken languages and proficiency levels."),
    ResponseSchema(name="certifications", description="List of certifications, courses, awards, or honors.")
]

output_parser = StructuredOutputParser.from_response_schemas(response_schemas)
format_instructions = output_parser.get_format_instructions()
print(format_instructions)


In [ ]:
cv_extraction_template = """
You are a CV/resume information extraction assistant.

Extract structured information from the CV below.

Rules:
1. Use ONLY information explicitly present in the CV.
2. Do NOT invent or assume information.
3. Keep professional work/internships separate from academic/personal projects.
4. If information is missing, use an empty string or empty list.
5. Return ONLY valid JSON and no explanation.
6. Follow these format instructions exactly:

{format_instructions}

Required logical structure:
{{
  "full_name": "",
  "email": "",
  "phone": "",
  "linkedin": "",
  "github": "",
  "profile": "",
  "education": [],
  "experience": [],
  "projects": [],
  "skills": [],
  "languages": [],
  "certifications": []
}}

CV TEXT:
----------------
{cv_text}
----------------
"""

prompt = cv_extraction_template.format(
    cv_text=cv_text,
    format_instructions=format_instructions
)
response = generate_text(prompt)
print(response)


In [ ]:
def extract_json(text):
    match = re.search(r"```json\s*(.*?)\s*```", text, re.DOTALL | re.IGNORECASE)
    if match:
        candidate = match.group(1).strip()
    else:
        start, end = text.find("{"), text.rfind("}")
        if start == -1 or end <= start:
            raise ValueError("No JSON object found.")
        candidate = text[start:end + 1]
    return json.loads(candidate)

result = extract_json(response)
print(json.dumps(result, indent=2, ensure_ascii=False))


In [ ]:
import os

print(os.listdir("/kaggle/input"))

In [ ]:
print(os.listdir("/kaggle/input/datasets/ahmedsuma/stream-app"))

In [ ]:
!pip install pyngrok

In [ ]:
pip install -q fastapi uvicorn requests

In [ ]:
!pip install -q streamlit

In [ ]:
!streamlit --version

In [ ]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import uvicorn
import threading

app = FastAPI(title="CV AI Inference Server")


class CVRequest(BaseModel):
    cv_text: str


class CVResponse(BaseModel):
    result: dict


@app.get("/")
def root():
    return {
        "status": "online",
        "service": "CV AI Inference Server"
    }


@app.get("/health")
def health():
    return {
        "status": "healthy",
        "model": model_name
    }


@app.post("/analyze", response_model=CVResponse)
def analyze_cv(request: CVRequest):

    try:
        # Build the same prompt used by the notebook
        prompt = cv_extraction_template.format(
            cv_text=request.cv_text,
            format_instructions=format_instructions
        )

        # Use the ALREADY LOADED Mistral model
        response = generate_text(
            prompt,
            max_new_tokens=1000
        )

        # Convert Mistral output into Python dictionary
        result = extract_json(response)

        return {
            "result": result
        }

    except Exception as e:
        raise HTTPException(
            status_code=500,
            detail=str(e)
        )


def run_api():
    uvicorn.run(
        app,
        host="0.0.0.0",
        port=8000,
        log_level="info"
    )


api_thread = threading.Thread(
    target=run_api,
    daemon=True
)

api_thread.start()

print("AI Inference Server started on port 8000")

In [ ]:
import requests

response = requests.get("http://127.0.0.1:8000/health")

print(response.status_code)
print(response.json())

In [ ]:
from pyngrok import ngrok

ngrok.set_auth_token("3KF09B1KAbSPafFD0zmXXhPLmB8_7P8ZcExV61WFBVpSiE6tn")

In [ ]:
import subprocess
import time

streamlit_process = subprocess.Popen(
    [
        "streamlit",
        "run",
        "/kaggle/input/datasets/ahmedsuma/stream-app/app.py",
        "--server.port",
        "8501",
        "--server.address",
        "0.0.0.0",
    ]
)

time.sleep(5)

print("Streamlit started.")
print("PID:", streamlit_process.pid)

In [ ]:
public_url = ngrok.connect(8501)
print(public_url)